# Дообучение эмбеддера v2 — исправленная версия

### Что исправлено по сравнению с v1

1. **Ложные негативы (главная причина деградации).** В датасете ~268 уникальных документов на 3000 запросов (≈11 запросов на документ). `MultipleNegativesRankingLoss` использует in-batch negatives: все позитивы других пар в батче = негативы. Если 2 запроса в батче указывают на один позитив → модель учится ОТТАЛКИВАТЬ правильный документ. **Решение:** дедупликация — оставляем 1 лучший запрос на уникальный позитив, плюс чистый hard negative. Это даёт ~268 чистых троек без ложных негативов.

2. **Слишком высокий LR → catastrophic forgetting.** 2e-5 на маленьком датасете с 3 эпохами — модель забывает общие знания. **Решение:** LR = 5e-6, 2 эпохи, увеличенный warmup.

3. **Eval-корпус слишком мал.** Evaluator строил корпус только из eval-split (~30 документов). **Решение:** в корпус eval попадают ВСЕ уникальные документы из всего датасета, а eval-запросы — hold-out подмножество.

4. **Добавлена стратегия 2:** если после дедупликации хочется использовать ВСЕ 3000 строк — переключаемся на `TripletLoss`, который не создаёт in-batch negatives и безопасен при дублирующихся позитивах.

## 0. Окружение

In [1]:
!nvidia-smi

Tue May 26 16:04:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U "sentence-transformers>=3.3.0" "transformers>=4.51.0" "datasets>=2.19.0" "accelerate>=0.30.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.6 MB/s eta 0:00:00


In [3]:
import os
import json
import random
import hashlib
import logging
from pathlib import Path
from collections import defaultdict

import torch
import numpy as np
import pandas as pd
from datasets import Dataset

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
    evaluation,
)
from sentence_transformers.training_args import BatchSamplers

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM, GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

Device: cuda
GPU: Tesla T4
VRAM, GB: 15.6


/tmp/ipykernel_855/908020164.py:14: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (
/tmp/ipykernel_855/908020164.py:14: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers import (
/tmp/ipykernel_855/908020164.py:21: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers


## 1. Модель и префиксы

In [4]:
BASE_MODEL = "intfloat/multilingual-e5-base"
OUTPUT_DIR = f"./finetuned-{BASE_MODEL.split('/')[-1]}-legal-ru-v2"

def get_prefix_fn(model_name: str):
    name = model_name.lower()
    if "e5" in name:
        return (lambda q: f"query: {q}", lambda d: f"passage: {d}")
    if "bge" in name and "m3" in name:
        return (lambda q: q, lambda d: d)
    if "qwen3-embedding" in name:
        instr = "Given a legal question in Russian, retrieve relevant passages from traffic regulations and laws"
        return (lambda q: f"Instruct: {instr}\nQuery: {q}", lambda d: d)
    if "rosberta" in name:
        return (lambda q: f"search_query: {q}", lambda d: f"search_document: {d}")
    return (lambda q: q, lambda d: d)

format_query, format_doc = get_prefix_fn(BASE_MODEL)
print("Query prefix:", format_query("тест"))
print("Doc prefix:",   format_doc("тест"))

Query prefix: query: тест
Doc prefix: passage: тест


## 2. Загрузка и дедупликация датасета

### Почему это критично

`MultipleNegativesRankingLoss` (MNRL) работает так: в батче из N пар `(q_i, p_i)` для каждого `q_i` все `p_j (j ≠ i)` считаются негативами. Если два запроса `q_1` и `q_2` в одном батче указывают на один и тот же документ `p`, то:
- для `q_1` документ `p` помечен как позитив, но одновременно (как позитив `q_2`) он попадает в негативы
- модель получает **противоречивый сигнал** и деградирует

**Решение:** оставить только 1 запрос на каждый уникальный документ. Выбираем самый длинный и информативный запрос.

In [5]:
DATA_PATH   = "/content/train-2.jsonl"   # <--- первый датасет
DATA_PATH_2 = "/content/train.jsonl"   # <--- второй датасет для сравнения

# Словарь для хранения результатов обоих прогонов
all_results = {}

# ========== Выбор стратегии ==========
# "mnrl_dedup"  — дедуплицируем позитивы (1 запрос на документ), используем MNRL. Меньше данных, но чистый сигнал.
# "triplet_full" — используем ВСЕ 3000 строк, но с TripletLoss (не создаёт in-batch negatives). Больше данных.
STRATEGY = "mnrl_dedup"   # <--- переключай тут

print(f"Стратегия: {STRATEGY}")
print(f"Датасет 1: {DATA_PATH}")
print(f"Датасет 2: {DATA_PATH_2}")


Стратегия: mnrl_dedup
Датасет 1: /content/train-2.jsonl
Датасет 2: /content/train.jsonl


In [6]:
# Загрузка через pandas (обход проблемы с кодировкой в load_dataset)
df = pd.read_json(DATA_PATH, lines=True, encoding="utf-8")
print(f"Загружено строк: {len(df)}")
print(f"Колонки: {list(df.columns)}")

# Считаем статистику дублирования
pos_counts = df["positive"].value_counts()
print(f"\nУникальных позитивов: {len(pos_counts)}")
print(f"Макс. запросов на 1 позитив: {pos_counts.max()}")
print(f"Среднее: {pos_counts.mean():.1f}")
print(f"\nТоп-5 дублированных позитивов (сколько запросов указывают на них):")
for pos_text, cnt in pos_counts.head(5).items():
    print(f"  {cnt}x : {pos_text[:120]}...")

Загружено строк: 3000
Колонки: ['query', 'positive', 'negative']

Уникальных позитивов: 743
Макс. запросов на 1 позитив: 12
Среднее: 4.0

Топ-5 дублированных позитивов (сколько запросов указывают на них):
  12x : Выезд на перекресток или пересечение проезжей части дороги в случае образовавшегося затора, который вынудил водителя ост...
  11x : Проезд на запрещающий сигнал светофора или на запрещающий жест регулировщика, за исключением случаев, предусмотренных ча...
  10x : За административные правонарушения, предусмотренные настоящей статьей, лица, осуществляющие предпринимательскую деятельн...
  10x : Нарушение правил проезда через железнодорожные переезды, за исключением случаев, предусмотренных частью 1 настоящей стат...
  10x : Движение на грузовом автомобиле с разрешенной максимальной массой более 3,5 тонны по автомагистрали далее второй полосы,...


In [7]:
if STRATEGY == "mnrl_dedup":
    # ============ ДЕДУПЛИКАЦИЯ ============
    # Для каждого уникального позитива оставляем 1 запрос.
    # Критерий: берём самый длинный запрос (обычно самый информативный)
    # и самый длинный негатив (самый hard).

    best_per_positive = {}
    for _, row in df.iterrows():
        pos_key = row["positive"]
        if pos_key not in best_per_positive:
            best_per_positive[pos_key] = row
        else:
            # Берём запрос, который длиннее и информативнее
            existing = best_per_positive[pos_key]
            if len(row["query"]) > len(existing["query"]):
                best_per_positive[pos_key] = row

    deduped = pd.DataFrame(list(best_per_positive.values()))
    print(f"После дедупликации: {len(deduped)} строк (было {len(df)})")
    print(f"Уникальных позитивов: {deduped['positive'].nunique()}")

    work_df = deduped
else:
    # ============ ВСЕ ДАННЫЕ (для TripletLoss) ============
    work_df = df.copy()
    print(f"Используем все {len(work_df)} строк с TripletLoss")

print(f"\nРазмер рабочего датасета: {len(work_df)}")

После дедупликации: 743 строк (было 3000)
Уникальных позитивов: 743

Размер рабочего датасета: 743


In [8]:
# Применяем префиксы
def apply_prefixes(row):
    return {
        "anchor":   format_query(row["query"]),
        "positive": format_doc(row["positive"]),
        "negative": format_doc(row["negative"]),
    }

work_prefixed = work_df.apply(apply_prefixes, axis=1, result_type="expand")

# Train/eval split — разделяем по ПОЗИТИВАМ, а не по строкам,
# чтобы eval содержал документы, которых модель не видела при обучении.
unique_positives = work_prefixed["positive"].unique()
np.random.shuffle(unique_positives)
n_eval = max(20, int(len(unique_positives) * 0.15))  # 15% позитивов в eval
eval_positives = set(unique_positives[:n_eval])
train_positives = set(unique_positives[n_eval:])

train_mask = work_prefixed["positive"].isin(train_positives)
eval_mask  = work_prefixed["positive"].isin(eval_positives)

train_df = work_prefixed[train_mask].reset_index(drop=True)
eval_df  = work_prefixed[eval_mask].reset_index(drop=True)

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
eval_ds  = Dataset.from_pandas(eval_df, preserve_index=False)

print(f"Train: {len(train_ds)} строк ({len(train_positives)} уник. позитивов)")
print(f"Eval:  {len(eval_ds)} строк ({len(eval_positives)} уник. позитивов)")
print(f"Пересечение позитивов train∩eval: {len(train_positives & eval_positives)} (должно быть 0)")

Train: 632 строк (632 уник. позитивов)
Eval:  111 строк (111 уник. позитивов)
Пересечение позитивов train∩eval: 0 (должно быть 0)


## 3. Eval: полный корпус

Корпус для `InformationRetrievalEvaluator` строим из **всех** уникальных документов (train + eval), а eval-запросы — только из eval-split. Это имитирует реальную RAG-систему: модель ищет по всему корпусу, а не по горстке eval-документов.

In [9]:
def build_full_corpus_evaluator(eval_data, all_data, name="legal-eval"):
    """
    Корпус = ВСЕ уникальные позитивы + негативы из всего датасета.
    Запросы = только из eval-split.
    """
    # 1) Собираем полный корпус
    corpus = {}  # doc_id -> text
    doc_text_to_id = {}  # text -> doc_id (дедупликация)

    def add_to_corpus(text):
        if text not in doc_text_to_id:
            doc_id = f"d{len(doc_text_to_id)}"
            doc_text_to_id[text] = doc_id
            corpus[doc_id] = text
        return doc_text_to_id[text]

    # Добавляем все документы из всего датасета
    for _, row in all_data.iterrows():
        add_to_corpus(row["positive"])
        add_to_corpus(row["negative"])

    # 2) Запросы и релевантности — только из eval
    queries = {}
    relevant_docs = {}

    for i, row in eval_data.iterrows():
        qid = f"q{i}"
        queries[qid] = row["anchor"]
        pos_id = doc_text_to_id[row["positive"]]
        relevant_docs[qid] = {pos_id}

    print(f"Eval: {len(queries)} запросов, корпус: {len(corpus)} документов")

    return evaluation.InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name=name,
        accuracy_at_k=[1, 3, 5, 10],
        precision_recall_at_k=[1, 3, 5, 10],
        mrr_at_k=[10],
        ndcg_at_k=[10],
        map_at_k=[10],
        show_progress_bar=True,
    )

ir_evaluator = build_full_corpus_evaluator(eval_df, work_prefixed)

Eval: 111 запросов, корпус: 853 документов


## 4. Бейзлайн

In [10]:
model = SentenceTransformer(BASE_MODEL, device=DEVICE)
model.max_seq_length = min(model.max_seq_length, 512)

print("Макс. длина:", model.max_seq_length)
print("Размерность эмбеддинга:", model.get_sentence_embedding_dimension())

baseline_metrics = ir_evaluator(model)
print("\n=== БЕЙЗЛАЙН ===")
for k in sorted(baseline_metrics.keys()):
    if any(m in k for m in ["ndcg@10", "mrr@10", "accuracy@1", "accuracy@5", "recall@10"]):
        print(f"  {k:55s} {baseline_metrics[k]:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:04<00:00,  4.92s/it]


=== БЕЙЗЛАЙН ===
  legal-eval_cosine_accuracy@1                            0.3063
  legal-eval_cosine_accuracy@10                           0.7027
  legal-eval_cosine_accuracy@5                            0.6577
  legal-eval_cosine_mrr@10                                0.4449
  legal-eval_cosine_ndcg@10                               0.5083
  legal-eval_cosine_recall@10                             0.7027


## 5. Лосс

Два варианта в зависимости от стратегии:

- **`mnrl_dedup`** — `MultipleNegativesRankingLoss`. Безопасен, потому что каждый позитив встречается только 1 раз → нет ложных негативов в батче.

- **`triplet_full`** — `TripletLoss`. Не использует in-batch negatives вообще — каждая тройка (anchor, positive, negative) обрабатывается независимо. Можно спокойно иметь дубли позитивов.

In [11]:
if STRATEGY == "mnrl_dedup":
    loss = losses.MultipleNegativesRankingLoss(model=model)
    print("Loss: MultipleNegativesRankingLoss (дедуплицированный датасет)")
elif STRATEGY == "triplet_full":
    loss = losses.TripletLoss(model=model, distance_metric=losses.TripletDistanceMetric.COSINE)
    print("Loss: TripletLoss (полный датасет, без in-batch negatives)")

Loss: MultipleNegativesRankingLoss (дедуплицированный датасет)


## 6. Параметры обучения

Ключевые отличия от v1:
- **LR = 5e-6** (было 2e-5) — мягкая адаптация без catastrophic forgetting
- **2 эпохи** (было 3) — меньше риск переобучения
- **warmup_ratio = 0.2** (было 0.1) — дольше прогреваемся
- **gradient_accumulation_steps = 4** с batch_size=8 → эффективный батч 32
- **eval каждые 50 шагов** (было 100) — датасет маленький, шагов мало

In [13]:
# Вычисляем количество шагов для адаптации eval_steps
effective_batch = 8 * 4  # per_device * accumulation
steps_per_epoch = max(1, len(train_ds) // effective_batch)
total_steps = steps_per_epoch * 2  # 2 эпохи
eval_steps = max(10, steps_per_epoch // 3)  # ~3 раза за эпоху

print(f"Размер train: {len(train_ds)}")
print(f"Эффективный батч: {effective_batch}")
print(f"Шагов/эпоха: {steps_per_epoch}")
print(f"Всего шагов: {total_steps}")
print(f"Eval каждые {eval_steps} шагов")

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,        # эффективный батч = 32
    learning_rate=5e-6,                   # ← ключевое: мягкий LR
    warmup_ratio=0.2,
    fp16=True,
    bf16=False,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=eval_steps,
    save_total_limit=2,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_legal-eval_cosine_ndcg@10",
    greater_is_better=True,
    report_to="none",
    gradient_checkpointing=False,
    seed=SEED,
)

Размер train: 632
Эффективный батч: 32
Шагов/эпоха: 19
Всего шагов: 38
Eval каждые 10 шагов


## 7. Обучение

In [14]:
torch.cuda.empty_cache()

In [15]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    loss=loss,
    evaluator=ir_evaluator,
)

trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Legal-eval Cosine Accuracy@1,Legal-eval Cosine Accuracy@3,Legal-eval Cosine Accuracy@5,Legal-eval Cosine Accuracy@10,Legal-eval Cosine Precision@1,Legal-eval Cosine Precision@3,Legal-eval Cosine Precision@5,Legal-eval Cosine Precision@10,Legal-eval Cosine Recall@1,Legal-eval Cosine Recall@3,Legal-eval Cosine Recall@5,Legal-eval Cosine Recall@10,Legal-eval Cosine Ndcg@10,Legal-eval Cosine Mrr@10,Legal-eval Cosine Map@10
10,1.794671,2.453307,0.342342,0.576577,0.657658,0.711712,0.342342,0.192192,0.131532,0.071171,0.342342,0.576577,0.657658,0.711712,0.529252,0.470231,0.470231
20,1.396658,1.739949,0.387387,0.612613,0.693694,0.783784,0.387387,0.204204,0.138739,0.078378,0.387387,0.612613,0.693694,0.783784,0.583912,0.520142,0.520142
30,1.027103,1.309348,0.396396,0.693694,0.783784,0.882883,0.396396,0.231231,0.156757,0.088288,0.396396,0.693694,0.783784,0.882883,0.642618,0.565244,0.565244
40,0.857465,1.037173,0.468468,0.783784,0.837838,0.918919,0.468468,0.261261,0.167568,0.091892,0.468468,0.783784,0.837838,0.918919,0.706891,0.637745,0.637745
50,0.683927,0.899784,0.513514,0.819820,0.873874,0.936937,0.513514,0.273273,0.174775,0.093694,0.513514,0.819820,0.873874,0.936937,0.740929,0.676712,0.676712
60,0.635221,0.853818,0.531532,0.819820,0.873874,0.927928,0.531532,0.273273,0.174775,0.092793,0.531532,0.819820,0.873874,0.927928,0.743687,0.683219,0.683219


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:01<00:00,  1.58s/it]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

TrainOutput(global_step=60, training_loss=1.0658408164978028, metrics={'train_runtime': 888.7101, 'train_samples_per_second': 2.133, 'train_steps_per_second': 0.068, 'total_flos': 0.0, 'train_loss': 1.0658408164978028, 'epoch': 3.0})

## 8. Сравнение

In [16]:
tuned_metrics = ir_evaluator(model)

print("\n=== СРАВНЕНИЕ ===")
print(f"{'metric':55s} {'baseline':>10s} {'tuned':>10s} {'delta':>10s}")
print("-" * 90)
for k in sorted(baseline_metrics.keys()):
    b = baseline_metrics[k]
    t = tuned_metrics.get(k, float("nan"))
    delta = t - b
    marker = "✅" if delta > 0.001 else ("❌" if delta < -0.001 else "  ")
    print(f"{marker} {k:53s} {b:10.4f} {t:10.4f} {delta:+10.4f}")

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:01<00:00,  1.51s/it]


=== СРАВНЕНИЕ ===
metric                                                    baseline      tuned      delta
------------------------------------------------------------------------------------------
✅ legal-eval_cosine_accuracy@1                              0.3063     0.5315    +0.2252
✅ legal-eval_cosine_accuracy@10                             0.7027     0.9279    +0.2252
✅ legal-eval_cosine_accuracy@3                              0.5676     0.8198    +0.2523
✅ legal-eval_cosine_accuracy@5                              0.6577     0.8739    +0.2162
✅ legal-eval_cosine_map@10                                  0.4449     0.6832    +0.2383
✅ legal-eval_cosine_mrr@10                                  0.4449     0.6832    +0.2383
✅ legal-eval_cosine_ndcg@10                                 0.5083     0.7437    +0.2353
✅ legal-eval_cosine_precision@1                             0.3063     0.5315    +0.2252
✅ legal-eval_cosine_precision@10                            0.0703     0.0928    +0.0225


In [17]:
# Сохраняем результаты первого датасета
all_results[DATA_PATH] = {
    "baseline": dict(baseline_metrics),
    "tuned": dict(tuned_metrics),
}
print(f"✅ Результаты для {DATA_PATH} сохранены")


✅ Результаты для /content/train-2.jsonl сохранены


## 9. Сохранение

In [19]:
FINAL_DIR = f"{OUTPUT_DIR}/final"
model.save(FINAL_DIR)
print("Сохранено в:", FINAL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Сохранено в: ./finetuned-multilingual-e5-base-legal-ru-v2/final


In [24]:
# from huggingface_hub import HubApi, Repository
import os
from transformers import AutoModel, AutoTokenizer
from google.colab import userdata

HF_TOKEN = userdata.get('HF')
repo_name = "maximmikhalevich/e5_large_for_law_RAG_2"

# model = AutoModel.from_pretrained("path/to/your/local/model/or/model-name")
tokenizer = AutoTokenizer.from_pretrained("/content/finetuned-multilingual-e5-base-legal-ru-v2/final")

model.push_to_hub(repo_name, token=HF_TOKEN)
tokenizer.push_to_hub(repo_name, token=HF_TOKEN)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpvcatkkr2/tokenizer.json:  98%|#########7| 16.7MB / 17.1MB            

  ...catkkr2/model.safetensors:   0%|          |  775kB / 1.11GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpgv3jtyy6/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

CommitInfo(commit_url='https://huggingface.co/maximmikhalevich/e5_large_for_law_RAG_2/commit/a2e182f2f0ae8990ed9b92f34ac1f7bc80c76e93', commit_message='Upload tokenizer', commit_description='', oid='a2e182f2f0ae8990ed9b92f34ac1f7bc80c76e93', pr_url=None, repo_url=RepoUrl('https://huggingface.co/maximmikhalevich/e5_large_for_law_RAG_2', endpoint='https://huggingface.co', repo_type='model', repo_id='maximmikhalevich/e5_large_for_law_RAG_2'), pr_revision=None, pr_num=None)

## 12. Повтор пайплайна на втором датасете

Перезагружаем модель с нуля (чтобы файн-тюн первого датасета не влиял), прогоняем тот же пайплайн на `DATA_PATH_2`.

In [29]:
torch.cuda.empty_cache()

# ===== Загрузка второго датасета =====
df2 = pd.read_json(DATA_PATH_2, lines=True, encoding="utf-8")
print(f"Датасет 2: {DATA_PATH_2}")
print(f"Загружено строк: {len(df2)}")

pos_counts2 = df2["positive"].value_counts()
print(f"Уникальных позитивов: {len(pos_counts2)}")
print(f"Макс. запросов на 1 позитив: {pos_counts2.max()}")

# ===== Дедупликация / выбор стратегии =====
if STRATEGY == "mnrl_dedup":
    best2 = {}
    for _, row in df2.iterrows():
        pk = row["positive"]
        if pk not in best2 or len(row["query"]) > len(best2[pk]["query"]):
            best2[pk] = row
    work_df2 = pd.DataFrame(list(best2.values()))
    print(f"После дедупликации: {len(work_df2)} строк")
else:
    work_df2 = df2.copy()
    print(f"Используем все {len(work_df2)} строк")

# ===== Префиксы + сплит =====
work_prefixed2 = work_df2.apply(apply_prefixes, axis=1, result_type="expand")

unique_pos2 = work_prefixed2["positive"].unique()
np.random.seed(SEED)
np.random.shuffle(unique_pos2)
n_eval2 = max(20, int(len(unique_pos2) * 0.15))
eval_pos2 = set(unique_pos2[:n_eval2])

train_df2 = work_prefixed2[~work_prefixed2["positive"].isin(eval_pos2)].reset_index(drop=True)
eval_df2  = work_prefixed2[work_prefixed2["positive"].isin(eval_pos2)].reset_index(drop=True)

train_ds2 = Dataset.from_pandas(train_df2, preserve_index=False)
eval_ds2  = Dataset.from_pandas(eval_df2, preserve_index=False)

print(f"Train: {len(train_ds2)}, Eval: {len(eval_ds2)}")

# ===== Evaluator =====
ir_evaluator2 = build_full_corpus_evaluator(eval_df2, work_prefixed2, name="legal-eval")

# ===== Свежая модель =====
model2 = SentenceTransformer(BASE_MODEL, device=DEVICE)
model2.max_seq_length = min(model2.max_seq_length, 512)

# ===== Бейзлайн =====
baseline2 = ir_evaluator2(model2)
print("\n=== БЕЙЗЛАЙН (датасет 2) ===")
for k in sorted(baseline2.keys()):
    if any(m in k for m in ["ndcg@10", "mrr@10", "accuracy@1", "recall@10"]):
        print(f"  {k:55s} {baseline2[k]:.4f}")


Датасет 2: /content/train.jsonl
Загружено строк: 1764
Уникальных позитивов: 346
Макс. запросов на 1 позитив: 8
После дедупликации: 346 строк
Train: 295, Eval: 51
Eval: 51 запросов, корпус: 346 документов


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:02<00:00,  2.22s/it]


=== БЕЙЗЛАЙН (датасет 2) ===
  legal-eval_cosine_accuracy@1                            0.2353
  legal-eval_cosine_accuracy@10                           0.6078
  legal-eval_cosine_mrr@10                                0.3431
  legal-eval_cosine_ndcg@10                               0.4058
  legal-eval_cosine_recall@10                             0.6078


In [31]:
# ===== Лосс =====
if STRATEGY == "mnrl_dedup":
    loss2 = losses.MultipleNegativesRankingLoss(model=model2)
else:
    loss2 = losses.TripletLoss(model=model2, distance_metric=losses.TripletDistanceMetric.COSINE)

# ===== Training args =====
effective_batch2 = 8 * 4
steps2 = max(1, len(train_ds2) // effective_batch2)
eval_steps2 = max(10, steps2 // 3)

OUTPUT_DIR_2 = OUTPUT_DIR + "-dataset2"

args2 = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR_2,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    warmup_ratio=0.2,
    fp16=True,
    bf16=False,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="steps",
    eval_steps=eval_steps2,
    save_strategy="steps",
    save_steps=eval_steps2,
    save_total_limit=2,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_legal-eval_cosine_ndcg@10",
    greater_is_better=True,
    report_to="none",
    gradient_checkpointing=False,
    seed=SEED,
)

# ===== Обучение =====
trainer2 = SentenceTransformerTrainer(
    model=model2,
    args=args2,
    train_dataset=train_ds2,
    eval_dataset=eval_ds2,
    loss=loss2,
    evaluator=ir_evaluator2,
)

trainer2.train()


Step,Training Loss,Validation Loss,Legal-eval Cosine Accuracy@1,Legal-eval Cosine Accuracy@3,Legal-eval Cosine Accuracy@5,Legal-eval Cosine Accuracy@10,Legal-eval Cosine Precision@1,Legal-eval Cosine Precision@3,Legal-eval Cosine Precision@5,Legal-eval Cosine Precision@10,Legal-eval Cosine Recall@1,Legal-eval Cosine Recall@3,Legal-eval Cosine Recall@5,Legal-eval Cosine Recall@10,Legal-eval Cosine Ndcg@10,Legal-eval Cosine Mrr@10,Legal-eval Cosine Map@10
10,2.031753,2.171868,0.294118,0.529412,0.627451,0.725490,0.294118,0.176471,0.125490,0.072549,0.294118,0.529412,0.627451,0.725490,0.503397,0.432540,0.432540
20,1.629037,1.763859,0.313725,0.607843,0.666667,0.784314,0.313725,0.202614,0.133333,0.078431,0.313725,0.607843,0.666667,0.784314,0.547647,0.471755,0.471755
30,1.420088,1.645485,0.372549,0.666667,0.725490,0.784314,0.372549,0.222222,0.145098,0.078431,0.372549,0.666667,0.725490,0.784314,0.588828,0.524891,0.524891


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:00<00:00,  1.40it/s]


...

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:00<00:00,  1.39it/s]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

TrainOutput(global_step=30, training_loss=1.6936258951822916, metrics={'train_runtime': 543.3002, 'train_samples_per_second': 1.629, 'train_steps_per_second': 0.055, 'total_flos': 0.0, 'train_loss': 1.6936258951822916, 'epoch': 3.0})

In [32]:
# ===== Сравнение датасет 2 =====
tuned2 = ir_evaluator2(model2)

print("\n=== СРАВНЕНИЕ (датасет 2) ===")
print(f"{'metric':55s} {'baseline':>10s} {'tuned':>10s} {'delta':>10s}")
print("-" * 90)
for k in sorted(baseline2.keys()):
    b = baseline2[k]
    t = tuned2.get(k, float("nan"))
    delta = t - b
    marker = "✅" if delta > 0.001 else ("❌" if delta < -0.001 else "  ")
    print(f"{marker} {k:53s} {b:10.4f} {t:10.4f} {delta:+10.4f}")

# Сохраняем результаты
all_results[DATA_PATH_2] = {"baseline": dict(baseline2), "tuned": dict(tuned2)}
print(f"\n✅ Результаты для {DATA_PATH_2} сохранены")


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:00<00:00,  1.29it/s]


=== СРАВНЕНИЕ (датасет 2) ===
metric                                                    baseline      tuned      delta
------------------------------------------------------------------------------------------
✅ legal-eval_cosine_accuracy@1                              0.2353     0.3725    +0.1373
✅ legal-eval_cosine_accuracy@10                             0.6078     0.7843    +0.1765
✅ legal-eval_cosine_accuracy@3                              0.4118     0.6667    +0.2549
✅ legal-eval_cosine_accuracy@5                              0.4902     0.7255    +0.2353
✅ legal-eval_cosine_map@10                                  0.3431     0.5249    +0.1818
✅ legal-eval_cosine_mrr@10                                  0.3431     0.5249    +0.1818
✅ legal-eval_cosine_ndcg@10                                 0.4058     0.5888    +0.1831
✅ legal-eval_cosine_precision@1                             0.2353     0.3725    +0.1373
✅ legal-eval_cosine_precision@10                            0.0608     0.0784

## 13. Итоговое сравнение двух датасетов

In [34]:
# Финальная таблица: оба датасета рядом
KEY_METRICS = [
    "legal-eval_cosine_accuracy@1",
    "legal-eval_cosine_accuracy@3",
    "legal-eval_cosine_accuracy@5",
    "legal-eval_cosine_accuracy@10",
    "legal-eval_cosine_mrr@10",
    "legal-eval_cosine_ndcg@10",
    "legal-eval_cosine_recall@10",
    "legal-eval_cosine_map@10",
]

paths = list(all_results.keys())
label1 = paths[0].split("/")[-1]
label2 = paths[1].split("/")[-1]

r1 = all_results[paths[0]]
r2 = all_results[paths[1]]

header = f"{'metric':40s} | {label1:>22s} | {label2:>22s}"
sub    = f"{'':<40s} | {'base → tuned (Δ)':>22s} | {'base → tuned (Δ)':>22s}"
print("=" * len(header))
print(header)
print(sub)
print("=" * len(header))

for m in KEY_METRICS:
    b1 = r1["baseline"].get(m, 0); t1 = r1["tuned"].get(m, 0); d1 = t1 - b1
    b2 = r2["baseline"].get(m, 0); t2 = r2["tuned"].get(m, 0); d2 = t2 - b2
    short = m.split("cosine_")[-1]
    col1 = f"{b1:.3f}→{t1:.3f} ({d1:+.3f})"
    col2 = f"{b2:.3f}→{t2:.3f} ({d2:+.3f})"
    # Лучший дельта получает стрелку
    best = "⬅" if d1 > d2 else ("➡" if d2 > d1 else " ")
    print(f"{short:40s} | {col1:>22s} | {col2:>22s}  {best}")

print("=" * len(header))
print("⬅ = датасет 1 лучше,  ➡ = датасет 2 лучше")


metric                                   |          train-2.jsonl |            train.jsonl
                                         |       base → tuned (Δ) |       base → tuned (Δ)
accuracy@1                               |   0.306→0.532 (+0.225) |   0.235→0.373 (+0.137)  ⬅
accuracy@3                               |   0.568→0.820 (+0.252) |   0.412→0.667 (+0.255)  ➡
accuracy@5                               |   0.658→0.874 (+0.216) |   0.490→0.725 (+0.235)  ➡
accuracy@10                              |   0.703→0.928 (+0.225) |   0.608→0.784 (+0.176)  ⬅
mrr@10                                   |   0.445→0.683 (+0.238) |   0.343→0.525 (+0.182)  ⬅
ndcg@10                                  |   0.508→0.744 (+0.235) |   0.406→0.589 (+0.183)  ⬅
recall@10                                |   0.703→0.928 (+0.225) |   0.608→0.784 (+0.176)  ⬅
map@10                                   |   0.445→0.683 (+0.238) |   0.343→0.525 (+0.182)  ⬅
⬅ = датасет 1 лучше,  ➡ = датасет 2 лучше


Проверка количества чанков в исследуемом датасете
--


In [18]:
from __future__ import annotations

import re
import uuid
from dataclasses import dataclass
from typing import Iterable, Any, Literal
from pydantic import BaseModel, Field


JsonDict = dict[str, Any]

class DocumentInput(BaseModel):
    id: str | None = None
    text: str = Field(min_length=1)
    metadata: JsonDict = Field(default_factory=dict)


class ChunkConfigInput(BaseModel):
    chunk_size: int | None = None
    chunk_overlap: int | None = None
    min_chunk_size: int | None = None


class ChunkInput(BaseModel):
    id: str
    doc_id: str
    text: str
    metadata: JsonDict = Field(default_factory=dict)
    chunk_index: int


class ChunkingConfig(BaseModel):
    chunk_size: int = 900
    chunk_overlap: int = 150
    min_chunk_size: int = 80
    separators: list[str] = ["\n\n", "\n", ". ", "! ", "? ", "; ", ", ", " "]



_SPACE_RE = re.compile(r"\s+")


@dataclass
class RecursiveTextChunker:
    config: ChunkingConfig

    def chunk_documents(
        self,
        documents: Iterable[DocumentInput],
        override: ChunkConfigInput | None = None,
    ) -> list[ChunkInput]:
        chunks: list[ChunkInput] = []
        for doc_pos, doc in enumerate(documents):
            doc_id = doc.id or f"doc_{doc_pos}_{uuid.uuid4().hex[:8]}"
            texts = self.split_text(doc.text, override)
            for i, chunk_text in enumerate(texts):
                chunks.append(
                    ChunkInput(
                        id=f"{doc_id}::chunk_{i}",
                        doc_id=doc_id,
                        text=chunk_text,
                        metadata=doc.metadata.copy(),
                        chunk_index=i,
                    )
                )
        return chunks

    def split_text(self, text: str, override: ChunkConfigInput | None = None) -> list[str]:
        chunk_size = override.chunk_size if override and override.chunk_size else self.config.chunk_size
        chunk_overlap = override.chunk_overlap if override and override.chunk_overlap is not None else self.config.chunk_overlap
        min_chunk_size = override.min_chunk_size if override and override.min_chunk_size else self.config.min_chunk_size

        text = self._normalize(text)
        if len(text) <= chunk_size:
            return [text] if text else []

        pieces = self._recursive_split(text, chunk_size, self.config.separators)
        return self._merge_pieces(pieces, chunk_size, chunk_overlap, min_chunk_size)

    def _recursive_split(self, text: str, chunk_size: int, separators: list[str]) -> list[str]:
        if len(text) <= chunk_size:
            return [text]
        if not separators:
            return [text[i : i + chunk_size] for i in range(0, len(text), chunk_size)]

        sep = separators[0]
        parts = text.split(sep)
        if len(parts) == 1:
            return self._recursive_split(text, chunk_size, separators[1:])

        result: list[str] = []
        for i, part in enumerate(parts):
            if not part:
                continue
            # Restore separator except when it is whitespace-only.
            restored = part + (sep if i < len(parts) - 1 and sep.strip() else "")
            if len(restored) > chunk_size:
                result.extend(self._recursive_split(restored, chunk_size, separators[1:]))
            else:
                result.append(restored)
        return result

    def _merge_pieces(self, pieces: list[str], chunk_size: int, chunk_overlap: int, min_chunk_size: int) -> list[str]:
        chunks: list[str] = []
        current = ""

        for piece in pieces:
            if not current:
                current = piece
                continue
            if len(current) + len(piece) <= chunk_size:
                current += piece
            else:
                if len(current.strip()) >= min_chunk_size:
                    chunks.append(current.strip())
                current = self._tail(current, chunk_overlap) + piece
                if len(current) > chunk_size * 1.2:
                    chunks.append(current[:chunk_size].strip())
                    current = self._tail(current, chunk_overlap)

        if current.strip() and len(current.strip()) >= min_chunk_size:
            chunks.append(current.strip())
        return chunks

    @staticmethod
    def _tail(text: str, n: int) -> str:
        if n <= 0:
            return ""
        return text[-n:]

    @staticmethod
    def _normalize(text: str) -> str:
        return _SPACE_RE.sub(" ", text.replace("\u00a0", " ")).strip()

In [28]:
# Создаем конфиг и чанкер
config = ChunkingConfig(chunk_size=900, chunk_overlap=150, min_chunk_size=80)
chunker = RecursiveTextChunker(config)

def count_chunks_for_dataframe(df: pd.DataFrame, text_columns: list[str]) -> pd.DataFrame:
    """
    Добавляет колонки с количеством чанков для указанных текстовых колонок
    """
    df_copy = df.copy()

    for col in text_columns:
        if col not in df_copy.columns:
            print(f"Предупреждение: колонка '{col}' не найдена в DataFrame")
            continue

        chunk_counts = []
        for text in df_copy[col]:
            if pd.isna(text) or text == "":
                chunk_counts.append(0)
            else:
                doc = DocumentInput(text=str(text))
                chunks = chunker.split_text(doc.text)
                chunk_counts.append(len(chunks))

        df_copy[f'{col}_chunk_count'] = chunk_counts

    # Добавляем общую статистику
    for col in text_columns:
        if f'{col}_chunk_count' in df_copy.columns:
            df_copy[f'{col}_total_chunks'] = df_copy[f'{col}_chunk_count'].sum()

    return df_copy

# Загружаем ваш DataFrame (предполагаем, что он уже загружен)
# df = pd.read_csv('your_file.csv') или df = ваш существующий DataFrame

# Применяем функцию ко всем трём колонкам
text_columns = ['anchor', 'positive', 'negative']
result_df = count_chunks_for_dataframe(train_df, text_columns)

# Выводим результаты
print("=" * 60)
print("СТАТИСТИКА ПО ЧАНКАМ")
print("=" * 60)

for col in text_columns:
    chunk_col = f'{col}_chunk_count'
    if chunk_col in result_df.columns:
        total_chunks = result_df[chunk_col].sum()
        avg_chunks = result_df[chunk_col].mean()
        max_chunks = result_df[chunk_col].max()
        min_chunks = result_df[chunk_col].min()
        non_zero = (result_df[chunk_col] > 0).sum()

        print(f"\nКолонка: {col}")
        print(f"  Всего чанков: {total_chunks}")
        print(f"  Среднее чанков на строку: {avg_chunks:.2f}")
        print(f"  Максимум чанков: {max_chunks}")
        print(f"  Минимум чанков: {min_chunks}")
        print(f"  Строк с текстом: {non_zero} из {len(df)}")

# Показываем первые несколько строк с результатами
print("\n" + "=" * 60)
print("ПРИМЕР РЕЗУЛЬТАТОВ (первые 5 строк)")
print("=" * 60)
display_columns = ['anchor', 'positive', 'negative',
                   'anchor_chunk_count', 'positive_chunk_count', 'negative_chunk_count']
existing_columns = [col for col in display_columns if col in result_df.columns]
print(result_df[existing_columns].head())

# Дополнительная статистика: распределение чанков
print("\n" + "=" * 60)
print("РАСПРЕДЕЛЕНИЕ ЧАНКОВ")
print("=" * 60)

for col in text_columns:
    chunk_col = f'{col}_chunk_count'
    if chunk_col in result_df.columns:
        print(f"\n{col}:")
        print(result_df[chunk_col].value_counts().sort_index().head(10))

СТАТИСТИКА ПО ЧАНКАМ

Колонка: anchor
  Всего чанков: 632
  Среднее чанков на строку: 1.00
  Максимум чанков: 1
  Минимум чанков: 1
  Строк с текстом: 632 из 3000

Колонка: positive
  Всего чанков: 666
  Среднее чанков на строку: 1.05
  Максимум чанков: 3
  Минимум чанков: 1
  Строк с текстом: 632 из 3000

Колонка: negative
  Всего чанков: 655
  Среднее чанков на строку: 1.04
  Максимум чанков: 3
  Минимум чанков: 1
  Строк с текстом: 632 из 3000

ПРИМЕР РЕЗУЛЬТАТОВ (первые 5 строк)
                                              anchor  \
0  query: Что обязан делать водитель по правилу «...   
1  query: Какая ответственность за невыполнение в...   
2  query: Что обязан делать водитель по правилу «...   
3  query: Что означают сигналы светофора (регулир...   
4  query: Какая статья КоАП РФ за движение по авт...   

                                            positive  \
0  passage: Аварийная световая сигнализация должн...   
1  passage: Невыполнение водителем транспортного ...   
2  pass